Load the data

In [1]:
import pandas as pd
from pathlib import Path

# Get the current directory
current_dir = Path.cwd()

# Load the CSV files
matches_file_path = current_dir.parent / 'data' / 'epic7_matches.csv'
hero_codes_file_path = current_dir.parent / 'data' / 'epic7_hero_codes.csv'

matches_df = pd.read_csv(matches_file_path)

matches_df.head()

,my_team_picks,enemy_team_picks,my_bans,enemy_bans,first_pick_team,winner
0,"[{""hero_code"": ""c2090"", ""pick_order"": 2}, {""he...","[{""hero_code"": ""c2106"", ""pick_order"": 1}, {""he...","[""c2066"", ""c2011""]","[""c2113"", ""c2011""]",enemy_team,my_team
1,"[{""hero_code"": ""c5110"", ""pick_order"": 1}, {""he...","[{""hero_code"": ""c2042"", ""pick_order"": 2}, {""he...","[""c2066"", ""c2011""]","[""c2113"", ""c2066""]",my_team,my_team
2,"[{""hero_code"": ""c5110"", ""pick_order"": 2}, {""he...","[{""hero_code"": ""c2113"", ""pick_order"": 1}, {""he...","[""c2066"", ""c2011""]","[""c2066"", ""c2090""]",enemy_team,enemy_team
3,"[{""hero_code"": ""c5110"", ""pick_order"": 1}, {""he...","[{""hero_code"": ""c2113"", ""pick_order"": 2}, {""he...","[""c2066"", ""c2011""]","[""c2066"", ""c2112""]",my_team,my_team
4,"[{""hero_code"": ""c5110"", ""pick_order"": 2}, {""he...","[{""hero_code"": ""c2090"", ""pick_order"": 1}, {""he...","[""c2066"", ""c2011""]","[""c2042"", ""c2066""]",enemy_team,my_team


Rearrange to winning and losing teams

In [2]:
# Iterate over each match and apply the rules
lst = []
for index, row in matches_df.iterrows():
    winner = row['winner']  # Adjust this if your column name is different
    
    # Map winner to my_team and loser to enemy_team
    first_pick_mapping = {'my_team':'winning_team', 'enemy_team':'losing_team'} if winner == 'my_team' else {'my_team':'losing_team', 'enemy_team':'winning_team'}
    mapping = {'winning_team':'my', 'losing_team':'enemy'} if winner == 'my_team' else {'winning_team':'enemy', 'losing_team':'my'}

    new_row = {
        'winning_team_picks': row[mapping['winning_team'] + '_team_picks'],
        'losing_team_picks': row[mapping['losing_team'] + '_team_picks'],
        'winning_team_bans': row[mapping['winning_team'] + '_bans'],
        'losing_team_bans': row[mapping['losing_team'] + '_bans'],
        'first_pick_team': first_pick_mapping[row['first_pick_team']] 
    }

    # Append the new row
    lst.append(new_row)

    
# Initialize a new DataFrame to store the rearranged data
rearranged_matches_df = pd.DataFrame(lst, columns=[
    'winning_team_picks',
    'losing_team_picks',
    'winning_team_bans',
    'losing_team_bans',
    'first_pick_team'
])

# Shuffle the matches
rearranged_matches_df = rearranged_matches_df.sample(frac=1).reset_index(drop=True)

rearranged_matches_df.head()

,winning_team_picks,losing_team_picks,winning_team_bans,losing_team_bans,first_pick_team
0,"[{""hero_code"": ""c2090"", ""pick_order"": 2}, {""he...","[{""hero_code"": ""c2106"", ""pick_order"": 1}, {""he...","[""c2066"", ""c2011""]","[""c2113"", ""c2011""]",losing_team
1,"[{""hero_code"": ""c5110"", ""pick_order"": 1}, {""he...","[{""hero_code"": ""c2042"", ""pick_order"": 2}, {""he...","[""c2066"", ""c2011""]","[""c2113"", ""c2066""]",winning_team
2,"[{""hero_code"": ""c2113"", ""pick_order"": 1}, {""he...","[{""hero_code"": ""c5110"", ""pick_order"": 2}, {""he...","[""c2066"", ""c2090""]","[""c2066"", ""c2011""]",winning_team
3,"[{""hero_code"": ""c5110"", ""pick_order"": 1}, {""he...","[{""hero_code"": ""c2113"", ""pick_order"": 2}, {""he...","[""c2066"", ""c2011""]","[""c2066"", ""c2112""]",winning_team
4,"[{""hero_code"": ""c5110"", ""pick_order"": 2}, {""he...","[{""hero_code"": ""c2090"", ""pick_order"": 1}, {""he...","[""c2066"", ""c2011""]","[""c2042"", ""c2066""]",losing_team


## convert the matches to snapshots and one hot encode

In [10]:
import numpy as np
import ast
import pandas as pd

# Load the hero codes mapping
hero_mapping_df = pd.read_csv(hero_codes_file_path, header=None, names=['code', 'name'])
hero_codes = hero_mapping_df['code'].tolist()

# Define the vocabulary
vocab = ['Null', 'Ban', 'Pick', 'winning_team', 'losing_team', 'c_null'] + hero_codes + [str(i) for i in range(11)]
vocab_size = len(vocab)

# Create a mapping from word to index for one-hot encoding
word_to_index = {word: idx for idx, word in enumerate(vocab)}
index_to_word = {idx: word for word, idx in word_to_index.items()}

# Function to convert JSON-like strings into lists of dictionaries
def parse_json_like_string(json_str):
    try:
        return ast.literal_eval(json_str)
    except ValueError:
        return []

# Function to create snapshots for predicting the hero code
def create_pick_snapshots(match):
    snapshots = []
    targets = []
    sequence = []

    # Parse the JSON-like strings
    winning_picks = parse_json_like_string(match['winning_team_picks'])
    losing_picks = parse_json_like_string(match['losing_team_picks'])
    winning_bans = parse_json_like_string(match['winning_team_bans'])
    losing_bans = parse_json_like_string(match['losing_team_bans'])

    # Add all prebans to the sequence
    for ban in winning_bans:
        sequence.extend(['Ban', '0', 'winning_team', ban if ban else 'c_null'])
    for ban in losing_bans:
        sequence.extend(['Ban', '0', 'losing_team', ban if ban else 'c_null'])

    # Combine and sort all picks
    all_picks = winning_picks + losing_picks
    all_picks_sorted = sorted(all_picks, key=lambda x: x['pick_order'])

    # Create snapshots and targets for picks 1 to 10
    for pick in all_picks_sorted:
        pick_order = int(pick['pick_order'])
        team = 'winning_team' if pick in winning_picks else 'losing_team'

        if 1 <= pick_order <= 10:
            # Create a snapshot before the current pick
            snapshot = sequence + ['Pick', str(pick_order), team]

            # Pad the snapshot to 56 words
            snapshot = snapshot[:56] + ['Null'] * max(0, 56 - len(snapshot))
            snapshots.append(snapshot)

            # Set the target to be only the hero code (4th word)
            targets.append(pick['hero_code'])

        # Add the current pick to the sequence
        sequence.extend(['Pick', str(pick_order), team, pick['hero_code']])

    return snapshots, targets

# Convert all matches into snapshots and targets
results = rearranged_matches_df.apply(create_pick_snapshots, axis=1)

# Flatten the list of snapshots and targets
snapshots = []
targets = []

for match_snapshots, match_targets in results:
    snapshots.extend(match_snapshots)
    targets.extend(match_targets)

# Function to format the decoded snapshot into lines with 4 words each
def format_decoded_snapshot(snapshot):
    lines = [' '.join(snapshot[i:i+4]) for i in range(0, len(snapshot), 4)]
    return '\n'.join(lines)

# Visualise the first few snapshots and their targets
def visualise_snapshots_and_targets(snapshots, targets, num_samples=5):
    for i in range(min(num_samples, len(snapshots))):
        # Decode the snapshot
        snapshot = snapshots[i]

        # Format the snapshot for better readability
        formatted_snapshot = format_decoded_snapshot(snapshot)

        # Display the snapshot and target
        print(f"Snapshot {i+1}:\n{formatted_snapshot}")
        print(f"\nTarget (Match Outcome): {targets[i]}")
        print("=" * 50)

# Call the visualisation function to check the first 5 samples
visualise_snapshots_and_targets(snapshots, targets, num_samples=20)

Snapshot 1:
Ban 0 winning_team c2066
Ban 0 winning_team c2011
Ban 0 losing_team c2113
Ban 0 losing_team c2011
Pick 1 losing_team Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null

Target (Match Outcome): c2106
Snapshot 2:
Ban 0 winning_team c2066
Ban 0 winning_team c2011
Ban 0 losing_team c2113
Ban 0 losing_team c2011
Pick 1 losing_team c2106
Pick 2 winning_team Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null

Target (Match Outcome): c2090
Snapshot 3:
Ban 0 winning_team c2066
Ban 0 winning_team c2011
Ban 0 losing_team c2113
Ban 0 losing_team c2011
Pick 1 losing_team c2106
Pick 2 winning_team c2090
Pick 3 winning_team Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Null
Null Null Null Nul

In [6]:
# Convert snapshots to NumPy array for efficient one-hot encoding
snapshots = np.array(snapshots)

# Use NumPy for efficient one-hot encoding of snapshots
def one_hot_encode_snapshots(snapshots, vocab_size, word_to_index):
    encoded = np.zeros((len(snapshots), 56, vocab_size), dtype=np.int32)
    for i, snapshot in enumerate(snapshots):
        indices = [word_to_index.get(word, word_to_index['Null']) for word in snapshot]
        encoded[i, np.arange(56), indices] = 1
    return encoded

# Efficient one-hot encoding for targets
def one_hot_encode_targets(targets, vocab_size, word_to_index):
    indices = [word_to_index.get(target, word_to_index['Null']) for target in targets]
    encoded = np.zeros((len(targets), vocab_size), dtype=np.int32)
    encoded[np.arange(len(targets)), indices] = 1
    return encoded

# One-hot encode snapshots and targets
one_hot_encoded_snapshots = one_hot_encode_snapshots(snapshots, vocab_size, word_to_index)
one_hot_encoded_targets = one_hot_encode_targets(targets, vocab_size, word_to_index)

# Output the shape of the encoded data for verification
print(f'One-hot encoded snapshots shape: {one_hot_encoded_snapshots.shape}')
print(f'One-hot encoded targets shape: {one_hot_encoded_targets.shape}')

One-hot encoded snapshots shape: (14890, 56, 356)
One-hot encoded targets shape: (14890, 356)


In [14]:
# Split the data into training, validation, and test sets using slicing
X_train = one_hot_encoded_snapshots
y_train = one_hot_encoded_targets

# Take 15% of the training data for the test set
test_size = int(0.15 * len(X_train))
X_test = X_train[:test_size]
y_test = y_train[:test_size]

# Update the training data to exclude the test set
X_train = X_train[test_size:]
y_train = y_train[test_size:]

# Output the shapes of the split data
print(f"Training set shape: {X_train.shape}, {y_train.shape}")
print(f"Test set shape: {X_test.shape}, {y_test.shape}")

Training set shape: (12657, 56, 356), (12657, 356)
Test set shape: (2233, 56, 356), (2233, 356)


In [15]:
from sklearn.model_selection import KFold

# Define the number of folds
n_folds = 5

# Initialize KFold
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Initialize lists to store results
fold_accuracies = []
best_val_accuracy = 0  # To keep track of the best validation accuracy

# Split the data into n folds
train_val_splits = list(kf.split(X_train))

# Directory to save the best model
best_model_dir = 'best_pick_model_v1.keras'


In [13]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense, BatchNormalization

# Loop over each fold for cross-validation
for fold, (train_index, val_index) in enumerate(train_val_splits):
    print(f"Training fold {fold + 1}/{n_folds}...")

    # Split the data into training and validation sets for this fold
    X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
    y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

    # Define the model
    model = Sequential()
    model.add(LSTM(512, return_sequences=True, input_shape=(56, vocab_size)))
    model.add(BatchNormalization())
    model.add(LSTM(512, return_sequences=True))
    model.add(BatchNormalization())
    model.add(LSTM(512, return_sequences=True))
    model.add(BatchNormalization())
    model.add(LSTM(512, return_sequences=False))
    model.add(Dropout(0.2))
    model.add(BatchNormalization())

    model.add(Dense(vocab_size, activation='softmax'))

    # Compile the model
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001) # initialize the optimizer with lowered learning rate (default = 0.001)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

    # Early stopping callback
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=6,
        restore_best_weights=True,
        verbose=1)

    # Learning rate reduction callback
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,        # Reduce the learning rate by a factor of 0.5
        patience=3,        # Wait for 3 epochs of no improvement before reducing
        min_lr=1e-6,       # Set a minimum learning rate
        verbose=1          # Print updates when learning rate is reduced
    )

    # Train the model
    history = model.fit(
        X_train_fold, y_train_fold,
        validation_data=(X_val_fold, y_val_fold),
        epochs=100, batch_size=64,
        callbacks=[early_stopping, reduce_lr]
    )

    # Evaluate the model on the validation set
    val_loss, val_accuracy = model.evaluate(X_val_fold, y_val_fold)
    print(f"Fold {fold + 1} Validation Accuracy: {val_accuracy * 100:.2f}%")
    fold_accuracies.append(val_accuracy)

    # Check if this is the best model so far
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        model.save(best_model_dir)  # Save the best model
        print(f"New best model saved with accuracy: {val_accuracy * 100:.2f}%")

# Calculate the average accuracy across all folds
average_accuracy = sum(fold_accuracies) / n_folds
print(f"Average Validation Accuracy: {average_accuracy * 100:.2f}%")

Training fold 1/5...


2024-10-22 10:22:40.455377: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
2024-10-22 10:22:40.455439: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 36.00 GB
2024-10-22 10:22:40.455449: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 13.50 GB
2024-10-22 10:22:40.455473: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2024-10-22 10:22:40.455490: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
/opt/miniconda3/envs/tf-gpu/lib/python3.9/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequen

Epoch 1/100


2024-10-22 10:22:42.801277: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


194/194 ━━━━━━━━━━━━━━━━━━━━ 28s 116ms/step - accuracy: 0.0763 - loss: 5.1969 - val_accuracy: 0.0806 - val_loss: 5.2167 - learning_rate: 1.0000e-04
Epoch 2/100
117/194 ━━━━━━━━━━━━━━━━━━━━ 7s 100ms/step - accuracy: 0.1139 - loss: 4.5210

KeyboardInterrupt: 

In [ ]:
from tensorflow.keras.models import load_model

# Load the best model
best_model = load_model(best_model_dir)

# Evaluate the best model on the test set
test_loss, test_accuracy = best_model.evaluate(X_test, y_test)
print(f'Test Accuracy of Best Model: {test_accuracy * 100:.2f}%')

In [119]:
# Make predictions on the test data
y_pred = model.predict(X_test)

# Convert the 'code' to 'name' mapping in hero_mapping_df to a dictionary
hero_code_to_name = hero_mapping_df.set_index('code')['name'].to_dict()

# Function to decode a one-hot encoded snapshot back to words
def decode_one_hot_snapshot(one_hot_snapshot, index_to_word):
    return [index_to_word[np.argmax(token)] for token in one_hot_snapshot]

# Function to replace hero codes with hero names in the decoded snapshot
def decode_snapshot_with_names(snapshot, hero_code_to_name):
    return [hero_code_to_name.get(word, word) for word in snapshot]

# Function to decode one-hot encoded hero code back to words
def decode_one_hot_hero_code(one_hot_vector, index_to_word):
    hero_code_index = np.argmax(one_hot_vector)  # Find the index of the predicted word
    return index_to_word.get(hero_code_index, 'Unknown')

# Format the decoded sequence into lines with 4 words each, excluding 'Null'
def format_decoded_snapshot(decoded_snapshot):
    
    # Remove 'Null' padding for easier reading
    filtered_sequence = [word for word in decoded_snapshot if word != 'Null']
    lines = [' '.join(filtered_sequence[i:i+4]) for i in range(0, len(filtered_sequence), 4)]
    return '\n'.join(lines)

# Visualise predictions and actual targets for a few examples
def visualise_single_word_predictions(X_test, y_test, y_pred, index_to_word, num_samples=5):
    for i in range(min(num_samples, len(X_test))):
        
        # Decode the one-hot encoded snapshot to words
        decoded_snapshot = decode_one_hot_snapshot(X_test[i], index_to_word)

        # Replace hero codes with hero names
        decoded_snapshot_with_names = decode_snapshot_with_names(decoded_snapshot, hero_code_to_name)

        # Format the decoded snapshot for better readability
        formatted_snapshot = format_decoded_snapshot(decoded_snapshot_with_names)

        # Decode the actual target (hero code)
        actual_hero = decode_one_hot_hero_code(y_test[i], index_to_word)

        # Decode the predicted target (hero code)
        predicted_hero = decode_one_hot_hero_code(y_pred[i], index_to_word)

        # Display the input, actual target, and predicted target
        print(f"Snapshot {i+1}:\n{formatted_snapshot}")
        print(f"\nActual Hero: {hero_code_to_name.get(actual_hero, actual_hero)}")
        print(f"Predicted Hero: {hero_code_to_name.get(predicted_hero, predicted_hero)}")
        print("=" * 50)

# Call the function to visualise the predictions on the test set
visualise_single_word_predictions(X_test, y_test, y_pred, index_to_word, num_samples=50)

47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step
Example 1:
Input Snapshot:
Ban 0 winning_team c1153 Ban 0 winning_team c2066 Ban 0 losing_team c1153 Ban 0 losing_team c1117 Pick 1 losing_team
Actual Hero Code: c5110
Predicted Hero Code: c5110
Example 2:
Input Snapshot:
Ban 0 winning_team c1153 Ban 0 winning_team c2066 Ban 0 losing_team c1153 Ban 0 losing_team c1117 Pick 1 losing_team c5110 Pick 2 winning_team
Actual Hero Code: c2042
Predicted Hero Code: c5110
Example 3:
Input Snapshot:
Ban 0 winning_team c1153 Ban 0 winning_team c2066 Ban 0 losing_team c1153 Ban 0 losing_team c1117 Pick 1 losing_team c5110 Pick 2 winning_team c2042 Pick 3 winning_team
Actual Hero Code: c2090
Predicted Hero Code: c2016
Example 4:
Input Snapshot:
Ban 0 winning_team c1153 Ban 0 winning_team c2066 Ban 0 losing_team c1153 Ban 0 losing_team c1117 Pick 1 losing_team c5110 Pick 2 winning_team c2042 Pick 3 winning_team c2090 Pick 4 losing_team
Actual Hero Code: c2011
Predicted Hero Code: c2016
Example 5:
Input Snapshot